In [1]:
import os
from dotenv import load_dotenv
from google import genai



In [ ]:

from ingest import load_data, build_index
from raghelper import RagBase



In [ ]:

load_dotenv()
client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))



In [ ]:

documents = load_data()
index = build_index(documents)



In [ ]:

assistant = RagBase(index=index, client=client)



In [ ]:
answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can join the course now.

You can start whenever you want, as the videos and GitHub materials are available. If you want to receive a certificate, you will need to submit your project while submissions are still being accepted and complete the course with a "live" cohort, as certificates are not awarded for self-paced mode. You can also start learning and submitting homework without formal registration.


In [ ]:
from elasticsearch import Elasticsearch
es_client = Elasticsearch("http://localhost:9200")

In [8]:
index_settings = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0
    },
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "section": {"type": "text"},
            "question": {"type": "text"},
            "course": {"type": "keyword"} 
        }
    }
}

index_name = "course-questions"

# This creates the physical index inside your running Docker container
es_client.indices.create(index=index_name, body=index_settings)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'course-questions'})

In [ ]:
from tqdm.auto import tqdm

for doc in tqdm(documents):
    es_client.index(index=index_name, document=doc)

print(f"Successfully indexed {len(documents)} documents into Elasticsearch!")

  0%|          | 0/1346 [00:00<?, ?it/s]

🎉 Successfully indexed 1346 documents into Elasticsearch!


In [10]:
query = "I just discovered the course. Can I join now?"

# Define the BM25 search structure matching Alexey's format
search_query = {
    "size": 5,
    "query": {
        "bool": {
            "must": {
                "multi_match": {
                    "query": query,
                    "fields": ["question^2", "text", "section"],
                    "type": "best_fields"
                }
            },
            "filter": {
                "term": {
                    "course": "llm-zoomcamp"
                }
            }
        }
    }
}

response = es_client.search(index=index_name, body=search_query)
print(response)

{'took': 934, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 57, 'relation': 'eq'}, 'max_score': 60.49162, 'hits': [{'_index': 'course-questions', '_id': 'zvNN154BqLIk7qLv8xS4', '_score': 60.49162, '_source': {'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}}, {'_index': 'course-questions', '_id': '7fNN154BqLIk7qLv8xTa', '_score': 19.91495, '_ignored': ['answer.keyword'], '_source': {'id': 'aa310de435', 'course': 'llm-zoomcamp', 'section': 'Module 1: RAG', 'question': 'Can I run the course locally instead of Codespaces?', 'answer': 'Yes. Codespaces is just the easiest way for everyone to start with the same environment.\n\nYou can run the course locally if you are comfortable setti

In [11]:
# 1. Parse out the nested documents list from Elasticsearch
result_docs = []

for hit in response['hits']['hits']:
    # Each matching row's content is stored inside the '_source' key
    doc = hit['_source']
    # We can also track the relevance score calculated by the BM25 algorithm
    doc['score'] = hit['_score']
    result_docs.append(doc)

# 2. View the top hit matching your question
import json
print(json.dumps(result_docs[0], indent=2))

{
  "id": "74eb249bbf",
  "course": "llm-zoomcamp",
  "section": "General Course-Related Questions",
  "question": "I just discovered the course. Can I still join?",
  "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions.",
  "score": 60.49162
}


In [12]:
# Force-reload the helper script with your new changes
import importlib
import raghelper
importlib.reload(raghelper)
from raghelper import RagBase

# Instantiate using your operational Elasticsearch client connection
production_assistant = RagBase(es_client=es_client, client=client)

# Execute the complete full-circle pipeline!
final_answer = production_assistant.rag("I just discovered the course. Can I join now?")
print(final_answer)

Yes, you can still join. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [13]:
production_assistant.rag("what is taj mahal?")

"I don't know."